# Chapter 36
## F-I Curves Under Pulsed Excitation
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter36.ipynb)

## About this chapter

This chapter compares firing-rate curves driven by periodic (pulsed)
excitation with the steady-current f-I curves of Chapter 17. Square
pulses provide an idealized input, and RTM (reduced traub-miles)
simulations show how pulse timing and amplitude select the spikes that
determine the observed rate.

A pulsed f-I curve counts response opportunities per pulse period rather
than a continuous rate: a cell may fire once, skip, or fire more than once
per pulse, so the curve can show steps or a saturating plateau instead of
a smooth rise. The idealized square-pulse schematic isolates this timing
effect before conductance-based RTM dynamics are added.

For pulse period $T$, the observed rate is $f=N_{\rm spikes}/T_{\rm
observation}$. Unlike a steady injected current, the drive here is
nonzero only during a narrow pulse interval; sweeping the pulse
amplitude produces the pulsed f-I curve, which we compare against the
constant-drive f-I curve of the same RTM cell.

See [`chapter36.md`](chapter36.md) for the full guide, including
suggested order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit

## Idealized F-I Curve

A schematic, model-free f-I curve: silent for $I<I_c$, then rising as
$f=\sqrt{I-I_c}$ above threshold. This is the qualitative shape later
sections compare pulsed and constant drive against.

In [ ]:
def plot_idealized_f_i_curve(i_c=0.0):
    """Schematic type-1-onset f-I curve: silent below i_c, f = sqrt(I - i_c) above."""
    fig, ax = plt.subplots(figsize=(6, 3))

    i_below = i_c + np.arange(-100, 1) / 100
    ax.plot(i_below, np.zeros_like(i_below), '-k', linewidth=4)

    i_above = i_c + np.arange(0, 101) / 100
    ax.plot(i_above, np.sqrt(i_above - i_c), '-k', linewidth=2)

    ax.set_ylabel('$f$')
    ax.text(i_c - 0.02, -0.1, '$I_c$')
    ax.axis([i_c - 1, i_c + 1, 0, 1])
    ax.set_xticks([])
    ax.set_yticks([])

    plt.tight_layout()
    plt.show()

interact(plot_idealized_f_i_curve, i_c=(-0.5, 0.5, 0.05));

## Square Pulses

A schematic recreation of the book's hand-drawn pulse train: a periodic
current of baseline $I_L$, pulse height $I_H$, pulse width $\epsilon$, and
period $T$. This is the idealized forcing signal used to motivate the
pulsed f-I curves below.

In [ ]:
def plot_square_pulses(T=3.0, epsilon=0.4, i_l=1.0, i_h=3.0):
    """Schematic recreation of a hand-drawn matlab figure (figure.graffle):
    a train of narrow square current pulses of height i_h above a baseline
    i_l, width epsilon, and period T."""
    fig, ax = plt.subplots(figsize=(8, 5))

    t = [0.]
    i = [i_l]
    for k in range(3):
        t0 = k * T
        t += [t0, t0, t0 + epsilon, t0 + epsilon]
        i += [i_l, i_h, i_h, i_l]
    t.append(3 * T - epsilon)
    i.append(i_l)

    ax.plot(t, i, '-k', linewidth=3)
    ax.annotate('', xy=(3 * T, 0), xytext=(-0.2, 0),
                arrowprops=dict(arrowstyle='->', linewidth=2))
    ax.annotate('', xy=(0, i_h + 1), xytext=(0, -0.1),
                arrowprops=dict(arrowstyle='->', linewidth=2))

    ax.set_xlim(-0.3, 3 * T)
    ax.set_ylim(-0.7, i_h + 1.2)
    ax.axis('off')

    xticks = [-0.15, epsilon + 0.15, T - 0.15, T + epsilon + 0.25, 2 * T - 0.15, 2 * T + epsilon + 0.25]
    xticklabels = ['0', r'$\epsilon$', '$T$', r'$T\!+\!\epsilon$', '$2T$', r'$2T\!+\!\epsilon$']
    for xt, label in zip(xticks, xticklabels):
        ax.text(xt, -0.6, label, ha='center', fontsize=14)
    for yt, label in zip([i_l, i_h], ['$I_L$', '$I_H$']):
        ax.text(-0.35, yt, label, ha='right', va='center', fontsize=16)

    plt.tight_layout()
    plt.show()

interact(plot_square_pulses, T=(1.0, 5.0, 0.25), epsilon=(0.1, 1.5, 0.1),
         i_l=(0.0, 2.0, 0.25), i_h=(1.0, 5.0, 0.25));

## RTM F-I Curve Under Pulsed Excitation (shared simulation code)

The two RTM examples below drive a single reduced Traub-Miles (RTM) cell
either with a constant current or with a periodic train of pulses of the
same time-average amplitude, and compare the resulting f-I curves. They
share the same RTM kernel and only differ in leak conductance `g_l`
(0.1 vs 0.2), so the per-timestep update, spike-counting sweeps, and pulse
shape function are written once here and reused by both.

Each F-I sweep integrates 1000 ms at `dt=0.01` ms for 201 drive values,
continuing from the previous drive's final state, so the per-timestep
update is `numba`-compiled (`@njit`); everything else (the pulse shape,
plotting) is plain NumPy.

In [ ]:
@njit
def _rtm_alpha_h(v):
    return 0.128 * math.exp(-(v + 50) / 18)


@njit
def _rtm_beta_h(v):
    return 4.0 / (1 + math.exp(-(v + 27) / 5))


@njit
def _rtm_alpha_n(v):
    return 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))


@njit
def _rtm_beta_n(v):
    return 0.5 * math.exp(-(v + 57) / 40)


@njit
def _rtm_m_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def _rtm_step(v, m, h, n, i_ext_old, i_ext_mid, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05):
    v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
             + g_l * (v_l - v) + i_ext_old) / c
    h_inc = _rtm_alpha_h(v) * (1 - h) - _rtm_beta_h(v) * h
    n_inc = _rtm_alpha_n(v) * (1 - n) - _rtm_beta_n(v) * n

    v_tmp = v + dt05 * v_inc
    m_tmp = _rtm_m_inf(v_tmp)
    h_tmp = h + dt05 * h_inc
    n_tmp = n + dt05 * n_inc

    v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
             + g_l * (v_l - v_tmp) + i_ext_mid) / c
    h_inc = _rtm_alpha_h(v_tmp) * (1 - h_tmp) - _rtm_beta_h(v_tmp) * h_tmp
    n_inc = _rtm_alpha_n(v_tmp) * (1 - n_tmp) - _rtm_beta_n(v_tmp) * n_tmp

    v_new = v + dt * v_inc
    m_new = _rtm_m_inf(v_new)
    h_new = h + dt * h_inc
    n_new = n + dt * n_inc
    return v_new, m_new, h_new, n_new


def _rtm_f_i_curve_constant_python(i_ext_values, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, m_steps):
    """Frequency from the interval between the 1st and 2nd spikes under a
    constant drive, continuing from the previous drive's final state; if
    two spikes don't occur within m_steps, frequency is taken as 0."""
    v, m, h, n = -70.0, _rtm_m_inf(-70.0), 0.7, 0.6
    dt05 = dt / 2
    f_vec = np.zeros(len(i_ext_values))

    for ijk in range(len(i_ext_values)):
        i_ext = i_ext_values[ijk]
        t_spike_1 = 0.0
        t_spike_2 = 0.0
        num_spikes = 0
        v_old = v
        f = 0.0
        for k in range(1, m_steps + 1):
            v, m, h, n = _rtm_step(v, m, h, n, i_ext, i_ext, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05)
            if v < -20 and v_old >= -20:
                t_spike = (k * dt * (20 + v_old) + (k - 1) * dt * (-20 - v)) / (v_old - v)
                num_spikes += 1
                if num_spikes == 1:
                    t_spike_1 = t_spike
                elif num_spikes == 2:
                    t_spike_2 = t_spike
                    f = 1000.0 / (t_spike_2 - t_spike_1)
                    break
            v_old = v
        f_vec[ijk] = f

    return f_vec


def _rtm_f_i_curve_constant_kernel(_cache={}):
    """Lazily njit-compile the constant-drive kernel and cache it on the
    function object (a plain module-level assignment would be stripped by
    the notebook-definitions test loader, which only keeps imports and
    def/class statements)."""
    if "jit" not in _cache:
        _cache["jit"] = njit(_rtm_f_i_curve_constant_python)
    return _cache["jit"]


def _rtm_f_i_curve_pulsed_python(i_ext_values, shape_store, t_final, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, m_steps):
    """Firing rate (spike count over the fixed t_final window) under a
    periodic pulsed drive of the given time-average amplitude, continuing
    from the previous drive's final state."""
    v, m, h, n = -70.0, _rtm_m_inf(-70.0), 0.7, 0.6
    dt05 = dt / 2
    f_vec = np.zeros(len(i_ext_values))

    for ijk in range(len(i_ext_values)):
        i_ext = i_ext_values[ijk]
        num_spikes = 0
        v_old = v
        for k in range(1, m_steps + 1):
            i_old = i_ext * shape_store[k - 1]
            i_mid = i_ext * (shape_store[k - 1] + shape_store[k]) / 2
            v, m, h, n = _rtm_step(v, m, h, n, i_old, i_mid, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05)
            if v < -20 and v_old >= -20:
                num_spikes += 1
            v_old = v
        f_vec[ijk] = num_spikes / t_final * 1000.0

    return f_vec


def _rtm_f_i_curve_pulsed_kernel(_cache={}):
    """Lazily njit-compile the pulsed-drive kernel; see
    _rtm_f_i_curve_constant_kernel for why this isn't a module-level
    assignment."""
    if "jit" not in _cache:
        _cache["jit"] = njit(_rtm_f_i_curve_pulsed_python)
    return _cache["jit"]


def rtm_pulse_shape(t, alpha=1.0, period=25.0, n_avg=2000):
    """The input pulses always have the same shape, with temporal average 1;
    the caller applies the amplitude."""
    idx = np.arange(n_avg)
    ave = np.mean(np.exp(alpha * np.cos(np.pi * idx / n_avg) ** 2) - 1)
    return (np.exp(alpha * np.cos(np.pi * t / period) ** 2) - 1) / ave


def _compute_rtm_f_i_curves(g_l, i_ext_values=None, c=1.0, g_k=80.0, g_na=100.0,
                             v_k=-100.0, v_na=50.0, v_l=-67.0,
                             alpha=1.0, period=25.0, dt=0.01, t_final=1000.0,
                             use_numba=True):
    """F-I curves under a constant and under a pulsed drive, one entry per
    drive in i_ext_values; shared by both RTM_F_I_CURVE_PULSED_EXCITATION
    variants (they only differ in g_l)."""
    if i_ext_values is None:
        i_ext_values = np.linspace(0.0, 2.0, 201)
    m_steps = round(t_final / dt)

    constant_kernel = _rtm_f_i_curve_constant_kernel() if use_numba else _rtm_f_i_curve_constant_python
    pulsed_kernel = _rtm_f_i_curve_pulsed_kernel() if use_numba else _rtm_f_i_curve_pulsed_python

    f_vec_constant = constant_kernel(i_ext_values, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, m_steps)

    t = np.arange(m_steps + 1) * dt
    shape_store = rtm_pulse_shape(t, alpha=alpha, period=period)
    f_vec_pulsed = pulsed_kernel(i_ext_values, shape_store, t_final, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, m_steps)

    return f_vec_constant, f_vec_pulsed, i_ext_values


def plot_rtm_f_i_curves_pulsed_excitation(f_vec_constant, f_vec_pulsed, i_ext_vec):
    plt.figure(figsize=(7, 5))
    plt.plot(i_ext_vec, f_vec_constant, '.r', markersize=8, label='constant drive')
    plt.plot(i_ext_vec, f_vec_pulsed, '.b', markersize=8, label='pulsed drive')
    plt.xlabel(r'$I$ [$\mu$A/cm$^2$]')
    plt.ylabel('$f$')
    plt.legend()
    plt.tight_layout()
    plt.show()

### RTM F-I Curve, Pulsed Excitation

`g_l = 0.1`: computes and plots the constant-drive and pulsed-drive f-I
curves for the base RTM cell. The pulsed curve mode-locks to the 25 ms
pulse period and shows a wide plateau near $f=40$ Hz.

In [ ]:
def compute_rtm_f_i_curves_pulsed_excitation(i_ext_values=None, use_numba=True, **kwargs):
    return _compute_rtm_f_i_curves(0.1, i_ext_values=i_ext_values, use_numba=use_numba, **kwargs)

f_vec_constant, f_vec_pulsed, i_ext_vec = compute_rtm_f_i_curves_pulsed_excitation()
plot_rtm_f_i_curves_pulsed_excitation(f_vec_constant, f_vec_pulsed, i_ext_vec)

### RTM F-I Curve, Pulsed Excitation 2

Same comparison with `g_l = 0.2`. Comparing this against the previous
figure shows how much of the plateau shape is intrinsic to the pulse
timing rather than the particular leak conductance.

In [ ]:
def compute_rtm_f_i_curves_pulsed_excitation_2(i_ext_values=None, use_numba=True, **kwargs):
    return _compute_rtm_f_i_curves(0.2, i_ext_values=i_ext_values, use_numba=use_numba, **kwargs)

f_vec_constant_2, f_vec_pulsed_2, i_ext_vec_2 = compute_rtm_f_i_curves_pulsed_excitation_2()
plot_rtm_f_i_curves_pulsed_excitation(f_vec_constant_2, f_vec_pulsed_2, i_ext_vec_2)